# Customer Purchase Propensity Model

### Overview

This project builds a purchase propensity model to predict the likelihood that a customer will make a purchase within the next 30 days. The model is developed on _Databricks_ using a synthetic dataset designed to mirror real-world retail customer behavior. 

### Use Case

Retailers and e-commerce businesses frequently need to identify which customers are most likely to buy in the near term. A propensity model enables:

* **Targeted marketing campaigns** - focus outreach on high-propensity customers
* **Promotional budget optimization** - reduce spend on customers likely to buy organically
* **Personalization** - tailor messaging, offers, and channel mix by propensity score
* **Churn prevention** - identify low-propensity, high-value customers for re-engagement

### Dataset 

The dataset is fully synthetic and contains one row per customer, representing a point-in-time snapshot of **behavioral**, **demographic**, and **transactional** signals.

#### Feature Reference

Column Name|Description
-|-
Customer ID|Unique customer identifier
Customer Name|Synthetic customer name
Region|Geographic sales region
Home Store|Assigned home store
Segment|Customer value/behavior segment
Age Band|Synthetic age group
Income Band|Synthetic household income band
Loyalty Tier|Loyalty program tier
Preferred Channel|Most-used shopping channel
Favorite Category|Most-purchased category
Tenure Months|Months since first purchase
Last Purchase Date|Most recent purchase date
Days Since Last Purchase|Recency feature
Orders 12M|Trailing 12-month order frequency
Orders 90D|Trailing 90-days order frequency
Avg Order Value|Average historical order value
Total Spend 12M|Trailing 12-month customer spend
Gross Margin|Average customer gross margin percentage
Returns 12M|Returns in the last 12 months
Return Rate|Returns divided by orders
Promo Redemptions 12M|Count of promotions used in the last 12 months
Email Opens 30D|Email opens in the last 30 days
Email Clicks 30D|Email clicks in the last 30 days
Site Visits 30D|web/app sessions in the last 30 days
Cart Adds 30D|Product cart additions in the last 30 days
Abandoned Carts 30D|Abandoned carts in the last 30 days
Loyalty Points|Current loyalty point balance
Complaint Count 12M|Customer service complaints in the last 12 months
Nearest Store Distance Miles|Distance to nearest stores
Synthetic True Propensity|Synthetic probability used to create label
Purchased Next 30D|1 means customer purchased in the next 30 days
Next 30D Spend|Synthetic spend amount for purchasers in next 30 days

#### Feature Groups

The features naturally fall into the following 5 dimensions:

* **Recency** - `Days Since Last Purchase`
* **Frequency** - `Orders 12M`, `Orders 90D`, `Promo Redemptions 12M`
* **Monetary** - `Avg Order Value`, `Total Spend 12M`, `Gross Margin`, `Loyalty Points`
* **Engagement** - `Email Opens/Clicks 30D`, `Site Visits 30D`, `Cart Adds 30D`, `Abandoned Carts 30D`
* **Customer Profile** - `Region`, `Segment`, `Age Band`, `Income Band`, `Loyalty Tier`, `Preferred Channel`, `Tenure Months`, `Nearest Store Distance Miles`

## Import Packages

In [0]:
import math
import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## Read Data

In [0]:
# Read data from sql table
train_df = spark.read.table("workspace.default.synthetic_propensity_model_training_data").toPandas()
test_df = spark.read.table("workspace.default.synthetic_propensity_model_scoring_data").toPandas()

In [0]:
# Table descriptions
train_df.describe()

In [0]:
train_df.isna().sum()

## Numerical Features

In [0]:
# Numerical Features
numerical_features = [
    "Tenure Months",
    "Days Since Last Purchase",
    "Orders 12M",
    "Orders 90D",
    "Returns 12M",
    "Promo Redemptions 12M",
    "Email Opens 30D",
    "Email Clicks 30D",
    "Site Visits 30D",
    "Cart Adds 30D",
    "Abandoned Carts 30D",
    "Loyalty Points",
    "Complaint Count 12M",
    "Nearest Store Distance Miles",
    "Avg Order Value",
    "Total Spend 12M",
    "Gross Margin",
    "Returns 12M",
    "Return Rate"
]

# Convert to numeric
for col in numerical_features:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')  

In [0]:
# Distribution of numerical features
max_cols = 3 
total_plots = len(numerical_features)

n_rows = math.ceil(total_plots / max_cols)

fig = make_subplots(
    rows=n_rows, 
    cols=max_cols, 
    subplot_titles=numerical_features
)

for i, col in enumerate(numerical_features):
    clean_data = train_df[col].dropna()
    
    if clean_data.empty:
        print(f"Skipping {col}: No data.")
        continue
        
    grid_row = (i // max_cols) + 1
    grid_col = (i % max_cols) + 1
    
    sub_fig = ff.create_distplot([clean_data], [col], show_hist=True, show_rug=False)
    
    for trace in sub_fig.data:
        fig.add_trace(trace, row=grid_row, col=grid_col)

fig.update_layout(
    title_text="<b>Numerical Features Distribution</b>", 
    title_x=0.5,
    title_y=0.99,
    title_font=dict(size=24),
    height=350 * n_rows, 
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.show()

## Categorical Features

In [0]:
# Categorical Features
categorical_features = [
    col for col in train_df.columns if col not in numerical_features and col not in ["Customer ID", "Customer Name", "Synthetic True Propensity", "Purchased Next 30D", "Next 30D Spend", "Last Purchase Date"]
]

print(categorical_features)

In [0]:
# Categorical features distribution
THEME_COLOR = '#5b9ac8'

max_cols = 3  
total_plots = len(categorical_features)
n_rows = math.ceil(total_plots / max_cols)

fig = make_subplots(
    rows=n_rows, 
    cols=max_cols, 
    subplot_titles=categorical_features
)

for i, col in enumerate(categorical_features):
    counts = train_df[col].value_counts()
    
    if counts.empty:
        continue
        
    grid_row = (i // max_cols) + 1
    grid_col = (i % max_cols) + 1
    
    fig.add_trace(
        go.Bar(
            x=counts.index, 
            y=counts.values, 
            name=col,
            marker_color=THEME_COLOR 
        ),
        row=grid_row, col=grid_col
    )

fig.update_layout(
    title_text="<b>Categorical Feature Distributions</b>", 
    title_x=0.5,
    title_y=0.99,
    title_font=dict(size=24),
    height=350 * n_rows,  
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_xaxes(tickangle=45)
fig.show()


## Feature Encoding